#Datei verkleinern

In [ ]:
import pandas as pd

# 1. Datensatz laden (nur die Spalten, die wir wirklich brauchen)
cols_to_keep = ['track_popularity', 'acousticness', 'danceability', 'energy',
                'valence', 'tempo', 'loudness', 'speechiness', 'instrumentalness']
df = pd.read_csv('../data/master_dataset_enriched.csv', usecols=cols_to_keep)

# 2. Definiere, was ein "Hit" ist (z.B. Popularität über 75)
threshold = 75
df['is_hit'] = (df['track_popularity'] >= threshold).astype(int)

# 3. Den Datensatz aufteilen
hits = df[df['is_hit'] == 1]
non_hits = df[df['is_hit'] == 0]

print(f"Anzahl Hits: {len(hits)}")
print(f"Anzahl Nicht-Hits vor Kürzung: {len(non_hits)}")

# 4. Balancing & Verkleinerung
# Wir nehmen ALLE Hits und genau so viele zufällige Nicht-Hits
non_hits_sampled = non_hits.sample(n=len(hits), random_state=42)

# 5. Zusammenfügen zum finalen Trainings-Set
df_final = pd.concat([hits, non_hits_sampled]).sample(frac=1).reset_index(drop=True)

print(f"Finale Größe des Datensatzes: {len(df_final)}")

In [ ]:
import pandas as pd

# 1. Datensatz laden (nur die Spalten, die wir wirklich brauchen)
cols_to_keep = ['track_popularity', 'acousticness', 'danceability', 'energy',
                'valence', 'tempo', 'loudness', 'speechiness', 'instrumentalness']
df = pd.read_csv('../data/master_dataset_enriched.csv', usecols=cols_to_keep)

# 2. Definiere, was ein "Hit" ist (z.B. Popularität über 60, der neue Schwellenwert)
new_threshold = 50 # Using the new threshold proposed earlier
df['is_hit'] = (df['track_popularity'] >= new_threshold).astype(int)

# 3. Den Datensatz aufteilen
hits = df[df['is_hit'] == 1]
non_hits = df[df['is_hit'] == 0]

print(f"Anzahl Hits mit neuem Schwellenwert ({new_threshold}): {len(hits)}")
print(f"Anzahl Nicht-Hits vor Kürzung: {len(non_hits)}")

# 4. Balancing & Verkleinerung
# Wir nehmen ALLE Hits und so viele zufällige Nicht-Hits, dass der finale Datensatz 10.000 Einträge hat
# The number of non-hits to sample is 10000 - the number of hits
non_hits_to_sample = max(0, 10000 - len(hits)) # Ensure non_hits_to_sample is not negative
non_hits_sampled = non_hits.sample(n=non_hits_to_sample, random_state=42)

# 5. Zusammenfügen zum finalen Trainings-Set
df_final = pd.concat([hits, non_hits_sampled]).sample(frac=1).reset_index(drop=True)

print(f"Finale Größe des Datensatzes: {len(df_final)}")

# 6. Speichern des finalen Datensatzes
df_final.to_csv('../data/final_processed_dataset.csv', index=False)
print("Finaler Datensatz als '../data/final_processed_dataset.csv' gespeichert.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the 'track_popularity' column
track_popularity = pd.read_csv('../data/master_dataset_enriched.csv', usecols=['track_popularity'])

# 2. Create a histogram to visualize its distribution
plt.figure(figsize=(10, 6))
sns.histplot(track_popularity['track_popularity'], bins=50, kde=True)

# 3. Add appropriate labels and title
plt.title('Distribution of Track Popularity')
plt.xlabel('Track Popularity')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)

# 4. Display the plot
plt.show()

In [ ]:
import pandas as pd

# 1. Load the full dataset (only relevant columns for defining hits)
cols_to_keep = ['track_popularity']
df_full = pd.read_csv('../data/master_dataset_enriched.csv', usecols=cols_to_keep)

# 2. Define the new 'hit' threshold
new_threshold = 50
df_full['is_hit'] = (df_full['track_popularity'] >= new_threshold).astype(int)

# 3. Recalculate the number of hits and non-hits
new_hits_count = df_full[df_full['is_hit'] == 1].shape[0]
new_non_hits_count = df_full[df_full['is_hit'] == 0].shape[0]

print(f"New Threshold for Hits: {new_threshold}")
print(f"Number of Hits with new threshold: {new_hits_count}")
print(f"Number of Non-Hits with new threshold: {new_non_hits_count}")

In [ ]:
import pandas as pd

# 1. Spalten laden (jetzt inkl. release_year und speechiness für Filter)
cols_to_keep = ['track_popularity', 'acousticness', 'danceability', 'energy',
                'valence', 'tempo', 'loudness', 'speechiness', 'instrumentalness', 'release_year']
df = pd.read_csv('../data/master_dataset_enriched.csv', usecols=cols_to_keep)

# --- NEU: FILTER-SCHRITT ---
# Nur moderne Songs (ab 2000)
df = df[df['release_year'] >= 2000]

# Nur echte Songs (keine Podcasts/Hörbücher)
df = df[df['speechiness'] < 0.5]

# Keine reinen Instrumental-Tracks (optional, je nach Ziel)
df = df[df['instrumentalness'] < 0.5]
# ---------------------------

# 2. Definiere Hit
new_threshold = 50
df['is_hit'] = (df['track_popularity'] >= new_threshold).astype(int)

# 3. Den Datensatz aufteilen
hits = df[df['is_hit'] == 1]
non_hits = df[df['is_hit'] == 0]

# 4. Sauberes 50/50 Balancing auf 10.000 Einträge
# Wir nehmen 5.000 Hits und 5.000 Nicht-Hits
n_per_class = 5000

if len(hits) < n_per_class:
    n_per_class = len(hits) # Falls wir weniger als 5000 Hits nach dem Filtern haben

hits_sampled = hits.sample(n=n_per_class, random_state=42)
non_hits_sampled = non_hits.sample(n=n_per_class, random_state=42)

# 5. Zusammenfügen & Mischen
df_final = pd.concat([hits_sampled, non_hits_sampled]).sample(frac=1).reset_index(drop=True)

# Release Year brauchen wir jetzt nicht mehr fürs Training
df_final = df_final.drop(columns=['release_year'])

print(f"Finale Größe: {len(df_final)} (50% Hits, 50% Nicht-Hits)")
df_final.to_csv('../data/final_processed_dataset_V2.csv', index=False)